# 🎓 Lecture Transcriber

Transcribes all your lecture recordings at once on Google's GPU, sorts them into your courses using your timetable, and gives you **one readable transcript per lecture**.

**Before you start (once):** in Google Drive, create a folder **`Vorlesungen`**, with a folder **`Inbox`** inside it. Also put your TraiNex timetable (`.ics`) into `Vorlesungen`.

**Each time:**
1. Upload recordings into `Vorlesungen/Inbox`. Any mix works:
   - the zip from the app (*Lecture library → Send to Colab…*)
   - GoodNotes recordings (exported as `.mp3`, or the whole `.goodnotes` document)
   - any other audio files (`.m4a`, `.mp3`, `.wav`, …)
2. Here: **Runtime → Run all** (Laufzeit → Alle ausführen). Allow access to Google Drive when asked.
3. When it's done, the transcripts download as a zip. They're also saved in `Vorlesungen/Results`.
   In the app, *Lecture library → Import from Colab…* files them into your library.

If Google disconnects you halfway, just **Run all** again: finished recordings are skipped.

In [ ]:
#@title ⚙️ Settings
DRIVE_FOLDER = "Vorlesungen"  #@param {type:"string"}
MODEL = "large-v3"  #@param ["large-v3", "large-v3-turbo"]
#@markdown `large-v3` = most accurate · `large-v3-turbo` = ~3× faster, slightly less accurate
LANGUAGE = "de"  #@param ["de", "en", "auto"]
#@markdown Language for recordings whose language isn't known (app recordings keep their course's language)

In [ ]:
#@title 1 · Setup (GPU, code, Google Drive)
import os, subprocess, sys

# GPU?
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("❌ No GPU. Menu: Runtime → Change runtime type → T4 GPU → Save, then Run all again.")
print("✓ GPU:", gpu.stdout.strip())

# Code: the transcriber + the app's sorting logic (github.com/RKIein/whisper)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)
if os.path.isdir("/content/whisper"):
    subprocess.run(["git", "-C", "/content/whisper", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/RKIein/whisper", "/content/whisper"], check=True)
sys.path.insert(0, "/content/whisper")
print("✓ Code ready")

# Google Drive
from google.colab import drive
drive.mount("/content/drive")
BASE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(f"{BASE}/Inbox", exist_ok=True)
inbox = [f for f in os.listdir(f"{BASE}/Inbox") if not f.startswith(".") and f != "Done"]
ics = [f for f in os.listdir(BASE) if f.lower().endswith(".ics")]
print(f"✓ Drive: {len(inbox)} file(s) in {DRIVE_FOLDER}/Inbox")
print(f"✓ Timetable: {', '.join(ics)}" if ics else
      f"⚠ No timetable (.ics) in {DRIVE_FOLDER} — recordings will go to “Unsorted”.")
if not inbox:
    print(f"\n→ Upload recordings into {DRIVE_FOLDER}/Inbox first, then Run all again.")

In [ ]:
#@title 2 · Load the speech model
import glob, importlib
import colab_batch

def load_engine():
    try:
        engine = colab_batch.WhisperEngine(MODEL, device="cuda")
        engine.transcribe(colab_batch.np.zeros(16000, dtype="float32"), "de", None)   # test run
        return engine
    except Exception as e:
        if "cudnn" not in str(e).lower() and "cublas" not in str(e).lower():
            raise
        # Colab sometimes lacks the CUDA libraries faster-whisper needs — install and retry
        print("Installing CUDA libraries (one moment)…")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"], check=True)
        import ctypes, nvidia.cublas.lib, nvidia.cudnn.lib
        for lib_dir in (nvidia.cublas.lib.__path__[0], nvidia.cudnn.lib.__path__[0]):
            for so in sorted(glob.glob(f"{lib_dir}/*.so*")):
                try:
                    ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass
        return colab_batch.WhisperEngine(MODEL, device="cuda")

engine = load_engine()
print(f"✓ {MODEL} loaded on the GPU")

In [ ]:
#@title 3 · Transcribe everything in the Inbox
zip_path = colab_batch.run(BASE, engine, language=None if LANGUAGE == "auto" else LANGUAGE)

In [ ]:
#@title 4 · Download the transcripts
from google.colab import files
if zip_path:
    print(f"Also saved in Drive: {DRIVE_FOLDER}/Results/{os.path.basename(zip_path)}")
    files.download(zip_path)
else:
    print("Nothing new — upload recordings into the Inbox and Run all again.")